# Road-context one-time offline workflow

This notebook is the one-time data-build, grouped-evaluation, and model-selection workflow for road context. It is **not** part of the live navigation loop and it never feeds CAN/reference data into runtime fusion.

Run sections 1-2 once the paired raw recordings are available. Stop after section 2 until a versioned `RoadGraph` and trajectory-level offline candidate file are available. Run sections 3-5 only after that handoff. Do not export or wire a runtime artifact until both grouped evaluation gates pass.

In [1]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import json
import sys

# Jupyter often starts in the notebook directory rather than the repository root.
# Keep the explicit path first so this notebook works from any kernel directory
# on this project machine; replace it only if the repository is moved.
_project_root_candidates = (
    Path(r'E:/dead reckoning'),
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
)
PROJECT_ROOT = next(
    (candidate for candidate in _project_root_candidates
     if (candidate / 'src' / 'idr_backend').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Set PROJECT_ROOT to the repository containing src/idr_backend.')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd

from idr_backend.evaluation.replay import load_raw_replay_journey
from idr_backend.map_matching.graph import RoadGraph
from idr_backend.road_context.datasets import (
    RoadContextSourceDatasetConfig,
    build_road_context_source_dataset,
)
from idr_backend.road_context.offline_matching import (
    OfflineRoadMatchCandidate,
    OfflineRoadMatchConfig,
    build_matched_road_context_dataset,
)
from idr_backend.road_context.graph_features import build_road_context_edge_features
from idr_backend.road_context.features import build_road_context_feature_dataset
from idr_backend.road_context.experiments import (
    run_road_class_empirical_quantile_experiment,
    run_road_context_quantile_experiment,
)
from idr_backend.road_context.lightgbm_quantile import (
    RoadContextLightGBMConfig,
    fit_road_context_lightgbm_quantile_model,
)
from idr_backend.road_context.splits import (
    RoadContextSplitConfig,
    build_directed_edge_holdout_split_plan,
    build_journey_holdout_split_plan,
)

# Set this to the folder that contains raw/S-<journey>.csv and raw/V-<journey>.csv.
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DIRECTORY = DATA_ROOT / 'raw'
RUN_ID = 'road_context_v1'
OUTPUT_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'road_context' / RUN_ID
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

SOURCE_PATH = OUTPUT_DIRECTORY / 'source_rows.parquet'
SOURCE_AUDIT_PATH = OUTPUT_DIRECTORY / 'source_audits.json'
CANDIDATES_PATH = OUTPUT_DIRECTORY / 'offline_match_candidates.parquet'
MATCHED_PATH = OUTPUT_DIRECTORY / 'matched_rows.parquet'
FEATURES_PATH = OUTPUT_DIRECTORY / 'feature_rows.parquet'

if not RAW_DIRECTORY.exists():
    print(f'Raw recordings not found at {RAW_DIRECTORY}. Set DATA_ROOT before running Section 1.')
print('Output directory:', OUTPUT_DIRECTORY)

RuntimeError: Open this notebook with the repository root as the working directory.

## 1. Build the sparse source table — run once now

This loads paired phone/CAN recordings, quality-filters phone GNSS, samples at two-second cadence, and writes CAN speed only as an offline target. It does not run the EKF or a velocity model.

In [ ]:
journey_ids = sorted(
    sensor_path.name[2:-4]
    for sensor_path in RAW_DIRECTORY.glob('S-*.csv')
    if (RAW_DIRECTORY / f'V-{sensor_path.name[2:-4]}.csv').exists()
)
if not journey_ids:
    raise RuntimeError(f'No paired S-/V- recordings found in {RAW_DIRECTORY}.')

journeys = [load_raw_replay_journey(RAW_DIRECTORY, journey_id) for journey_id in journey_ids]
source_dataset = build_road_context_source_dataset(
    journeys,
    config=RoadContextSourceDatasetConfig(
        sample_period_s=2.0,
        match_position_source='phone_gnss',
    ),
)
source_dataset.frame.to_parquet(SOURCE_PATH, index=False)
SOURCE_AUDIT_PATH.write_text(
    json.dumps([asdict(audit) for audit in source_dataset.audits], indent=2),
    encoding='utf-8',
)

print('Journeys:', len(journey_ids))
print('Source rows:', len(source_dataset.frame))
display(pd.DataFrame([asdict(audit) for audit in source_dataset.audits]))

## 2. Offline matcher handoff — do not run until available

The project deliberately has matching **contracts**, not a concrete OSM graph loader or trajectory-matcher command. Supply one versioned `RoadGraph` and `offline_match_candidates.parquet` generated by a trajectory-level matcher. The candidate file must contain every candidate probability for each source `(journey_id, timestamp_ns)` and these columns: `journey_id`, `timestamp_ns`, `graph_id`, `edge_id`, `travel_direction`, `osm_way_id`, `candidate_rank`, `posterior`, `lateral_error_m`, and optional `heading_error_deg`.

Do not replace this with nearest-edge snapping. That would corrupt the training labels.

In [ ]:
# Replace this assignment with the graph loader for the exact OSM extract used
# to create CANDIDATES_PATH. The graph and candidate graph_id must agree.
graph: RoadGraph | None = None

if graph is None:
    raise RuntimeError(
        'STOP: load the versioned RoadGraph here, then rerun this cell and later cells.'
    )

assert CANDIDATES_PATH.exists(), (
    'STOP: write trajectory-level matcher candidates to ' + str(CANDIDATES_PATH)
)
candidate_frame = pd.read_parquet(CANDIDATES_PATH)
required_candidate_columns = {
    'journey_id', 'timestamp_ns', 'graph_id', 'edge_id', 'travel_direction',
    'osm_way_id', 'candidate_rank', 'posterior', 'lateral_error_m',
}
missing_candidate_columns = required_candidate_columns - set(candidate_frame.columns)
assert not missing_candidate_columns, sorted(missing_candidate_columns)

offline_candidates = tuple(
    OfflineRoadMatchCandidate(
        journey_id=str(row.journey_id),
        timestamp_ns=int(row.timestamp_ns),
        graph_id=str(row.graph_id),
        edge_id=str(row.edge_id),
        travel_direction=str(row.travel_direction),
        osm_way_id=str(row.osm_way_id),
        candidate_rank=int(row.candidate_rank),
        posterior=float(row.posterior),
        lateral_error_m=float(row.lateral_error_m),
        heading_error_deg=(
            None
            if 'heading_error_deg' not in candidate_frame or pd.isna(row.heading_error_deg)
            else float(row.heading_error_deg)
        ),
    )
    for row in candidate_frame.itertuples(index=False)
)
print('Candidate rows:', len(offline_candidates), '| graph:', graph.metadata.graph_id)

## 3. Build the immutable matched feature table — run once after section 2

This applies confidence/margin/geometry acceptance, derives legal directed-edge OSM features, and writes the only table allowed to enter model selection.

In [ ]:
matched_dataset = build_matched_road_context_dataset(
    source_dataset,
    offline_candidates,
    config=OfflineRoadMatchConfig(graph_id=graph.metadata.graph_id),
)
edge_features = build_road_context_edge_features(graph)
feature_dataset = build_road_context_feature_dataset(matched_dataset, edge_features)

matched_dataset.frame.to_parquet(MATCHED_PATH, index=False)
feature_dataset.frame.to_parquet(FEATURES_PATH, index=False)

print('Matched rows:', len(matched_dataset.frame))
print('Feature rows:', len(feature_dataset.frame))
display(feature_dataset.frame.head())

## 4. Compare models under both grouped holdouts — run after section 3

Run the empirical baseline first, then LightGBM with the same splits, weights, and OOF metric calculation. Do not select on training loss or on a single journey split.

In [ ]:
split_config = RoadContextSplitConfig(n_splits=5, seed=42)
journey_plan = build_journey_holdout_split_plan(feature_dataset, split_config)
edge_plan = build_directed_edge_holdout_split_plan(feature_dataset, split_config)

def fit_lightgbm(training_dataset, model_id, sample_weight):
    return fit_road_context_lightgbm_quantile_model(
        training_dataset,
        model_id=model_id,
        sample_weight=sample_weight,
        config=RoadContextLightGBMConfig(random_state=42, n_jobs=1),
    )

experiments = {
    'empirical_journey': run_road_class_empirical_quantile_experiment(
        dataset=feature_dataset, split_plan=journey_plan
    ),
    'empirical_directed_edge': run_road_class_empirical_quantile_experiment(
        dataset=feature_dataset, split_plan=edge_plan
    ),
    'lightgbm_journey': run_road_context_quantile_experiment(
        dataset=feature_dataset,
        split_plan=journey_plan,
        model_id_prefix='road_context_lightgbm_v1',
        fit_model=fit_lightgbm,
    ),
    'lightgbm_directed_edge': run_road_context_quantile_experiment(
        dataset=feature_dataset,
        split_plan=edge_plan,
        model_id_prefix='road_context_lightgbm_v1',
        fit_model=fit_lightgbm,
    ),
}

summary_rows = []
for experiment_name, result in experiments.items():
    metrics = result.evaluation_report.overall_metrics
    summary_rows.append({
        'experiment': experiment_name,
        'samples': metrics.sample_count,
        'p50_mae_kph': metrics.median_mae_kph,
        'pinball_loss_mps': metrics.mean_pinball_loss_mps,
        'p10_p90_coverage': metrics.observed_interval_coverage,
        'coverage_error': metrics.coverage_error,
        'mean_interval_width_kph': metrics.mean_interval_width_kph,
    })
    result.oof_predictions.to_parquet(
        OUTPUT_DIRECTORY / f'{experiment_name}_oof.parquet', index=False
    )

summary = pd.DataFrame(summary_rows).sort_values('experiment')
display(summary)
summary.to_csv(OUTPUT_DIRECTORY / 'grouped_evaluation_summary.csv', index=False)

## 5. Selection gate — do not export or integrate yet

Run this only after reading the OOF reports. The selected model must beat or match the empirical baseline on both split strategies, have p10-p90 coverage close to 0.80, and show no unsafe subgroup. A model-level pass still does **not** authorize EKF integration: it must first be loaded by the disabled `RoadContextCandidatePipeline`, observed in shadow mode, and then pass downstream blackout replay.